# Softmax Regression from Scratch (NumPy)

First, let's load our dataset (we use the Iris dataset) and set hyperparameters: num_epochs, batch_size and initialize the weight matrix $\Theta$:

$ \Theta = \begin{bmatrix}
\theta_{0}^{(0)} & \theta_{1}^{(0)} & \theta_{2}^{(0)} \\
\theta_{0}^{(1)} & \theta_{1}^{(1)} & \theta_{2}^{(1)} \\
... & ... & ... \\
\theta_{0}^{(K)} & \theta_{1}^{(K)} & \theta_{2}^{(K)} \\
\end{bmatrix}$

Where:
* $\theta_{i}^{(j)}$ - weight i for class j
* $K$ - num of classes

In [62]:
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = iris.target.to_numpy()

m = X.shape[0]

num_epochs = 5000
matrix_weight = np.array([np.random.randn(1, 3)[0] for _ in range(len(np.unique(y)))])
matrix_weight

array([[ 0.74034082, -1.03049894,  0.30501416],
       [-0.025749  , -0.34955074, -0.43451093],
       [-1.50146883,  1.06278813, -0.82325264]])

Add a column of ones for the bias

In [63]:
X = np.c_[np.ones((m, 1)), X]
X[:5]

array([[1. , 1.4, 0.2],
       [1. , 1.4, 0.2],
       [1. , 1.3, 0.2],
       [1. , 1.5, 0.2],
       [1. , 1.4, 0.2]])

Train-test split:

In [64]:
test_size = 0.2
len_test = int(test_size * X.shape[0])
len_train = X.shape[0] - len_test

shuffle = np.random.permutation(X.shape[0])
X = X[shuffle, :]
y = y[shuffle]

X_train, y_train = X[:len_train], y[:len_train]
X_test, y_test = X[len_train:], y[len_train:]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((120, 3), (120,), (30, 3), (30,))

Use one hot to transform y

In [65]:
def one_hot(y):
    unique_values = len(np.unique(y))
    transformed_matrix = np.zeros((len(y), unique_values))
    transformed_matrix[list(range(len(y))), list(y)] = 1
    return transformed_matrix


In [66]:
y_train_ohe = one_hot(y_train)
y_train[:5], y_train_ohe[:5]

(array([0, 2, 1, 0, 2]),
 array([[1., 0., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [1., 0., 0.],
        [0., 0., 1.]]))

Scaling X_train:

In [67]:
mean_X_train = X_train[:, 1:].mean(axis=0, keepdims=True)
std_X_train = X_train[:, 1:].std(axis=0, keepdims=True)

X_train[:, 1:] = (X_train[:, 1:] - mean_X_train) / std_X_train
X_train[:5]

array([[ 1.        , -1.23713201, -1.31064082],
       [ 1.        ,  1.19088409,  1.35959514],
       [ 1.        ,  0.32373548,  0.15798896],
       [ 1.        , -1.35275183, -1.31064082],
       [ 1.        ,  0.90183455,  1.49310694]])

We will use a mini batch gradient descent

First, we evaluate scores. Score for one class k:

$s_{k}(x) = (\theta^{(k)})^T x$

For example, score for class zero: 

$s_{0}(x) = (\theta^{(0)})^T x$

For all classes:

$X \cdot \Theta^T = \begin{bmatrix}
x_{0}^{(0)} & x_{1}^{(0)} & x_{2}^{(0)} \\
x_{0}^{(1)} & x_{1}^{(1)} & x_{2}^{(1)} \\
... & ... & ... \\
x_{0}^{(m)} & x_{1}^{(m)} & x_{2}^{(m)} \\
\end{bmatrix}
\cdot 
\begin{bmatrix}
\theta_{0}^{(0)} & \theta_{0}^{(1)} & ... & \theta_{0}^{(K)} \\
\theta_{1}^{(0)} & \theta_{1}^{(1)} & ... & \theta_{1}^{(K)} \\
\theta_{2}^{(0)} & \theta_{2}^{(1)} & ... & \theta_{2}^{(K)} \\
\end{bmatrix} =
\begin{bmatrix}
x_0^{(0)}\theta_0^{(0)} + x_1^{(0)}\theta_1^{(0)} + x_2^{(0)}\theta_2^{(0)} & x_0^{(0)}\theta_0^{(1)} + x_1^{(0)}\theta_1^{(1)} + x_2^{(0)}\theta_2^{(1)} & \dots & x_0^{(0)}\theta_0^{(K)} + x_1^{(0)}\theta_1^{(K)} + x_2^{(0)}\theta_2^{(K)} \\
x_0^{(1)}\theta_0^{(0)} + x_1^{(1)}\theta_1^{(0)} + x_2^{(1)}\theta_2^{(0)} & x_0^{(1)}\theta_0^{(1)} + x_1^{(1)}\theta_1^{(1)} + x_2^{(1)}\theta_2^{(1)} & \dots & x_0^{(1)}\theta_0^{(K)} + x_1^{(1)}\theta_1^{(K)} + x_2^{(1)}\theta_2^{(K)} \\
\dots & \dots & \dots & \dots \\
x_0^{(m)}\theta_0^{(0)} + x_1^{(m)}\theta_1^{(0)} + x_2^{(m)}\theta_2^{(0)} & x_0^{(m)}\theta_0^{(1)} + x_1^{(m)}\theta_1^{(1)} + x_2^{(m)}\theta_2^{(1)} & \dots & x_0^{(m)}\theta_0^{(K)} + x_1^{(m)}\theta_1^{(K)} + x_2^{(m)}\theta_2^{(K)}
\end{bmatrix} =
 \begin{bmatrix}
s_{0}(x^{(0)}) & s_{1}(x^{(0)}) & ... & s_{K}(x^{(0)}) \\
s_{0}(x^{(1)}) & s_{1}(x^{(1)}) & ... & s_{K}(x^{(1)}) \\
... & ... & ... & ... \\
s_{0}(x^{(m)}) & s_{1}(x^{(m)}) & ... & s_{K}(x^{(m)}) \\
\end{bmatrix} =
\begin{bmatrix}
s(x^{(0)}) \\
s(x^{(1)}) \\
... \\
s(x^{(m)}) \\
\end{bmatrix}$

Then we estimate the probabilitiy $\hat{p}_{k}$ for each class:

$\hat{p}_{k} = \sigma(s(x))_{k} = \frac{exp(s_{k}(x))}{\sum_{j=0}^{K}exp(s_{j}(x))}$

For MGD we use cross entropy cost function:

$J(\Theta) = -\frac{1}{m}\sum_{i=0}^{m}\sum_{k=0}^{K}y_{k}^{(i)}log(\hat{p}_{k}^{(i)})$

The gradient vector of this cost function:

$\nabla_{\theta^{(k)}}J(\Theta) = \frac{1}{m}\sum_{i=0}^{m}(\hat{p}_{k}^{(i)} - y_{k}^{(i)})x^{(i)}$

In [68]:
import numpy as np

def evaluate_gradients(X, y, matrix_weight):
    scores = X @ matrix_weight.T # scores: (m x K)
    scores = scores - scores.max(axis=1, keepdims=True) # overflow protection
    exp_scores = np.exp(scores)
    proba = exp_scores / exp_scores.sum(axis=1, keepdims=True) # proba: (m x K), one hot y: (m x K)
    gradients = ((proba - y).T @ X) / X.shape[0] 
    # (proba - y): (m x K), (proba - y).T: (K x m), X: (m x n)
    # (proba - y).T @ X: (K x n)
    return gradients

In [69]:
t0, t1 = 200, 1000

def learning_schedule(t):
    return t0 / (t + t1)

In [70]:
alpha = 0.2 

best_val_accuracy = float("-inf")
best_matrix_weight = None

shuffled = np.random.permutation(X_train.shape[0])
X_shuffled = X_train[shuffled, :]
y_shuffled = y_train_ohe[shuffled]

valid_size = 0.2
len_valid = int(X_train.shape[0] * valid_size)
len_train = X_train.shape[0] - len_valid

X_train_shfld = X_shuffled[:len_train]
y_train_shfld = y_shuffled[:len_train]
X_valid_shfld = X_shuffled[len_train:]
y_valid_shfld = y_shuffled[len_train:]

for epoch in range(num_epochs):
    if epoch % 500 == 0:
        print(f"Epoch: {epoch}")

    shuffled = np.random.permutation(X_train_shfld.shape[0])
    X_train_shfld = X_train_shfld[shuffled, :]
    y_train_shfld = y_train_shfld[shuffled]

    gradients = (
        evaluate_gradients(X_train_shfld, y_train_shfld, matrix_weight)
        + np.c_[np.zeros(matrix_weight.shape[0]), 2 * alpha * matrix_weight[:, 1:] / X_train_shfld.shape[0]] 
    ) # with l2 regularization
    learning_rate = learning_schedule(epoch * X_train_shfld.shape[0])
    matrix_weight -= learning_rate * gradients

    # early stopping
    preds_valid = np.argmax(X_valid_shfld @ matrix_weight.T, axis=1)
    valid_accuracy = (preds_valid == np.argmax(y_valid_shfld, axis=1)).sum() / y_valid_shfld.shape[0]
    if valid_accuracy > best_val_accuracy:
        best_val_accuracy = valid_accuracy
        best_matrix_weight = matrix_weight.copy()

print(best_val_accuracy.round(2))

Epoch: 0
Epoch: 500
Epoch: 1000
Epoch: 1500
Epoch: 2000
Epoch: 2500
Epoch: 3000
Epoch: 3500
Epoch: 4000
Epoch: 4500
0.96


And accuracy in test set:

In [78]:
finally_matrix_weight = best_matrix_weight.copy()

X_test_tr = np.c_[X_test[:, 0], (X_test[:, 1:] - mean_X_train) / std_X_train]
test_preds = np.argmax(X_test_tr @ finally_matrix_weight.T, axis=1)
accuracy = (test_preds == y_test).sum() / y_test.shape[0]
print(f"Accuracy in test set: {accuracy.round(2)}")

Accuracy in test set: 0.97
